# Lab 17 — Integrated Capstone
Compare logistic regression, decision tree and a PyTorch MLP; then select a model. Streamlined Colab edition.

In [ ]:
import numpy as np,pandas as pd,torch,torch.nn as nn,torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,roc_auc_score,confusion_matrix
X,y=load_breast_cancer(return_X_y=True); y=(y==0).astype(int); Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y); sc=StandardScaler(); Xtrs=sc.fit_transform(Xtr).astype('float32'); Xtes=sc.transform(Xte).astype('float32')

In [ ]:
def metrics(name,yt,yp,pr):
 tn,fp,fn,tp=confusion_matrix(yt,yp).ravel(); return [name,accuracy_score(yt,yp),recall_score(yt,yp),tn/(tn+fp),precision_score(yt,yp),f1_score(yt,yp),roc_auc_score(yt,pr)]
rows=[]
log=LogisticRegression(max_iter=2000,class_weight='balanced').fit(Xtrs,ytr); rows.append(metrics('Logistic',yte,log.predict(Xtes),log.predict_proba(Xtes)[:,1]))
tree=DecisionTreeClassifier(max_depth=4,min_samples_leaf=5,class_weight='balanced',random_state=42).fit(Xtr,ytr); rows.append(metrics('Tree',yte,tree.predict(Xte),tree.predict_proba(Xte)[:,1]))

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model=nn.Sequential(nn.Linear(30,32),nn.ReLU(),nn.Dropout(.2),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,2)).to(device); opt=optim.Adam(model.parameters(),1e-3); loss_fn=nn.CrossEntropyLoss(); xt=torch.tensor(Xtrs).to(device); yt=torch.tensor(ytr).long().to(device)
for e in range(100): opt.zero_grad(); loss=loss_fn(model(xt),yt); loss.backward(); opt.step()
with torch.no_grad(): p=torch.softmax(model(torch.tensor(Xtes).to(device)),1)[:,1].cpu().numpy(); pred=(p>=.5).astype(int)
rows.append(metrics('MLP',yte,pred,p)); table=pd.DataFrame(rows,columns=['Model','Accuracy','Sensitivity','Specificity','Precision','F1','ROC-AUC']); print(table); print('Selected by ROC-AUC:',table.loc[table['ROC-AUC'].idxmax(),'Model'])